In [1]:
import os
os.environ["GDAL_DATA"] = r"C:\Users\WeiKitPhang\.conda\envs\python_base_1\Library\share\gdal"


import ee
import geemap
import geopandas as gpd
import pandas as pd

# --------------------------------------------------
# Initialize Earth Engine
# --------------------------------------------------
ee.Authenticate()
ee.Initialize()



In [2]:
shapefile_path = r"C:\Users\WeiKitPhang\Documents\UM\Writing\Paper 21 (Gua Musang Preliminary)\GIS\GM_shapefile.shp"

gdf = gpd.read_file(shapefile_path)

# Ensure CRS is correct
if gdf.crs is None:
    gdf.set_crs(epsg=4326, inplace=True)
else:
    gdf = gdf.to_crs(epsg=4326)

# Convert to Earth Engine object
roi = geemap.geopandas_to_ee(gdf)

print("Shapefile loaded successfully. CRS:", gdf.crs)

Shapefile loaded successfully. CRS: EPSG:4326


In [3]:
# ==========================================================
# 3. LOAD EXCEL → CONVERT TO EE FEATURECOLLECTION
# ==========================================================
excel_path = r"C:\Users\WeiKitPhang\Documents\UM\Writing\Paper 21 (Gua Musang Preliminary)\Data_processing\Gua Musang case data.xlsx"

df = pd.read_excel(excel_path, sheet_name="Week_data")

df["start_dt"] = pd.to_datetime(df["start_dt"])
df["end_dt"] = pd.to_datetime(df["end_dt"])

# Convert pandas → EE FeatureCollection
features = []

for _, row in df.iterrows():
    features.append(ee.Feature(None, {
        "start_dt": row["start_dt"].strftime("%Y-%m-%d"),
        "end_dt": row["end_dt"].strftime("%Y-%m-%d")
    }))

weeks_fc = ee.FeatureCollection(features)

print(df.columns)
print(df.dtypes)

df.head()

Index(['epid_wk', 'start_dt', 'end_dt', 'case_count', 'case_count_l1',
       'case_count_l2', 'case_count_l3', 'precipitation', 'temp_2m',
       'dewpoint_2m', 'rh', 'pop_smooth', 'nino4_smooth', 'soi_smooth',
       'mei_smooth', 'mjo_romi'],
      dtype='object')
epid_wk                   int64
start_dt         datetime64[ns]
end_dt           datetime64[ns]
case_count                int64
case_count_l1           float64
case_count_l2           float64
case_count_l3           float64
precipitation           float64
temp_2m                 float64
dewpoint_2m             float64
rh                      float64
pop_smooth              float64
nino4_smooth            float64
soi_smooth              float64
mei_smooth              float64
mjo_romi                float64
dtype: object


,epid_wk,start_dt,end_dt,case_count,case_count_l1,case_count_l2,case_count_l3,precipitation,temp_2m,dewpoint_2m,rh,pop_smooth,nino4_smooth,soi_smooth,mei_smooth,mjo_romi
0,1,2012-01-01,2012-01-07,0,NaN,NaN,NaN,14.226180,22.770521,20.195255,85.401271,92267.867995,27.371389,1.238931,-1.127599,1.316081
1,2,2012-01-08,2012-01-14,1,0.0,NaN,NaN,25.498182,21.521455,20.452478,93.636478,92295.045862,27.344722,1.134913,-1.069003,1.223776
2,3,2012-01-15,2012-01-21,0,1.0,0.0,NaN,22.195561,23.162632,21.444645,90.066162,92322.223958,27.321981,1.030957,-1.010461,1.010023
3,4,2012-01-22,2012-01-28,3,0.0,1.0,0.0,4.477910,22.807737,19.258090,80.391368,92349.402670,27.305940,0.927549,-0.952063,1.436547
4,5,2012-01-29,2012-02-04,0,3.0,0.0,1.0,4.351615,22.780616,20.860153,88.924436,92376.582567,27.299368,0.825330,-0.893937,1.516986


In [4]:
#--------------------------------------------
# 4. Load datasets
# --------------------------------------------
chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
era5 = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")

In [5]:
# ==========================================================
# 5. SERVER-SIDE FUNCTION (NO PYTHON LOOP)
# ==========================================================
def compute_week(feature):

    start = ee.Date(feature.get("start_dt"))
    end = ee.Date(feature.get('end_dt')).advance(1, 'day') #  advance(1, 'day') makes end date inclusive

    rain = chirps.filterDate(start, end).mean().select("precipitation")

    era = era5.filterDate(start, end).mean()

    temp = era.select("temperature_2m").subtract(273.15)
    dew = era.select("dewpoint_temperature_2m").subtract(273.15)

    img = rain.rename("rainfall") \
        .addBands(temp.rename("temp_2m_C")) \
        .addBands(dew.rename("dewpoint_2m_C"))

    stats = img.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi.geometry(),
        scale=5000,
        maxPixels=1e13
    )

    return feature.set({
        "rainfall_mm_per_day": stats.get("rainfall"),
        "temp_2m_C": stats.get("temp_2m_C"),
        "dewpoint_2m_C": stats.get("dewpoint_2m_C")
    })


In [6]:
# ==========================================================
# 6. APPLY FUNCTION SERVER-SIDE (KEY SPEED BOOST)
# ==========================================================
result_fc = weeks_fc.map(compute_week)

In [7]:
#==========================================================
# 7. EXPORT (ONE JOB ONLY)
# ==========================================================
task = ee.batch.Export.table.toDrive(
    collection=result_fc,
    description="weekly_climate_extraction",
    fileFormat="CSV"
)

task.start()

print("Export started. Check Earth Engine Tasks tab.")

Export started. Check Earth Engine Tasks tab.
